# __CLIP-Based Temporal Anomaly Inference__

This notebook implements a pipeline that processes videos from the UCF-Crime dataset, extracts semantic anomaly cues using CLIP, and produces both a detailed JSON file and a UCF-compatible annotation file.

## __Setup and Required Libraries__

In [ ]:
!pip install torch torchvision clip ffmpeg-python tqdm matplotlib opencv-python

In [ ]:
import os
import sys
import json
import argparse
import subprocess
import numpy as np
import torch
import clip
import cv2
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## __Define Helper Functions__

In [ ]:
def extract_frames(video_path, output_dir, fps=30):
    """
    Extract frames from a video using ffmpeg
    
    Args:
        video_path: Path to the input video
        output_dir: Directory to save extracted frames
        fps: Frames per second to extract
        
    Returns:
        Path to the directory containing extracted frames
    """
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frames_dir = os.path.join(output_dir, 'frames', video_name)
    os.makedirs(frames_dir, exist_ok=True)
    
    # Check if frames already exist
    if len(os.listdir(frames_dir)) > 0:
        print(f"Frames already exist in {frames_dir}. Skipping extraction.")
        return frames_dir
    
    # Extract frames using ffmpeg
    command = [
        'ffmpeg', '-i', video_path, 
        '-r', str(fps), 
        os.path.join(frames_dir, 'frame_%06d.jpg')
    ]
    
    subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(f"Extracted frames to {frames_dir}")
    
    return frames_dir

In [ ]:
def load_clip_model(model_name="ViT-B/32"):
    """
    Load CLIP model and preprocessing transform
    
    Args:
        model_name: Name of the CLIP model to load
        
    Returns:
        model: Loaded CLIP model
        preprocess: CLIP preprocessing transform
    """
    model, preprocess = clip.load(model_name, device=device)
    model.eval()  # Set model to evaluation mode
    return model, preprocess

In [ ]:
def compute_frame_embeddings(frames_dir, model, preprocess, batch_size=32):
    """
    Compute CLIP embeddings for all frames in the directory
    
    Args:
        frames_dir: Directory containing extracted frames
        model: CLIP model
        preprocess: CLIP preprocessing transform
        batch_size: Number of frames to process at once
        
    Returns:
        frame_embeddings: Dictionary mapping frame indices to embeddings
        frame_paths: Sorted list of frame paths
    """
    # Get sorted list of frame paths
    frame_paths = sorted([os.path.join(frames_dir, f) for f in os.listdir(frames_dir) 
                          if f.startswith('frame_') and f.endswith('.jpg')])
    
    frame_embeddings = {}
    
    # Process frames in batches
    for i in tqdm(range(0, len(frame_paths), batch_size), desc="Computing embeddings"):
        batch_paths = frame_paths[i:i+batch_size]
        batch_frames = []
        
        # Load and preprocess frames
        for frame_path in batch_paths:
            image = Image.open(frame_path)
            processed_image = preprocess(image)
            batch_frames.append(processed_image)
        
        # Stack frames into a batch tensor
        batch_tensor = torch.stack(batch_frames).to(device)
        
        # Compute embeddings
        with torch.no_grad():
            batch_embeddings = model.encode_image(batch_tensor)
            batch_embeddings = batch_embeddings / batch_embeddings.norm(dim=1, keepdim=True)
        
        # Store embeddings
        for j, frame_path in enumerate(batch_paths):
            frame_idx = int(os.path.basename(frame_path).split('_')[1].split('.')[0])
            frame_embeddings[frame_idx] = batch_embeddings[j].cpu()
    
    return frame_embeddings, frame_paths

In [ ]:
def compute_prompt_scores(frame_embeddings, model, prompts):
    """
    Compute cosine similarity scores between frame embeddings and text prompts
    
    Args:
        frame_embeddings: Dictionary mapping frame indices to embeddings
        model: CLIP model
        prompts: List of text prompts
        
    Returns:
        frame_scores: Dictionary mapping frame indices to prompt scores
    """
    # Encode prompts
    with torch.no_grad():
        text_inputs = clip.tokenize(prompts).to(device)
        text_embeddings = model.encode_text(text_inputs)
        text_embeddings = text_embeddings / text_embeddings.norm(dim=1, keepdim=True)
    
    frame_scores = {}
    prompt_keys = ["normal", "anomaly", "attack", "suspicious"]
    
    # Compute scores for each frame
    for frame_idx, embedding in tqdm(frame_embeddings.items(), desc="Computing scores"):
        # Calculate cosine similarity with each prompt
        similarities = 100 * (embedding.to(device) @ text_embeddings.T)
        
        # Map prompt indices to keys
        scores = {}
        for i, key in enumerate(prompt_keys):
            scores[key] = similarities[i].item()
        
        frame_scores[frame_idx] = scores
    
    return frame_scores

In [ ]:
def calculate_deviation(frame_scores):
    """
    Calculate anomaly deviation for each frame
    
    Args:
        frame_scores: Dictionary mapping frame indices to prompt scores
        
    Returns:
        frame_indices: Sorted list of frame indices
        deviation_values: Corresponding deviation values
    """
    frame_indices = sorted(frame_scores.keys())
    deviation_values = []
    
    for idx in frame_indices:
        scores = frame_scores[idx]
        normal_score = scores["normal"]
        anomaly_score = scores["anomaly"]
        attack_score = scores["attack"]
        suspicious_score = scores["suspicious"]
        
        # Calculate deviation as maximum difference from normal score
        deviation = max(
            anomaly_score - normal_score,
            attack_score - normal_score,
            suspicious_score - normal_score,
            0  # Ensure we don't go below 0
        )
        
        deviation_values.append(deviation)
    
    return frame_indices, deviation_values

In [ ]:
def detect_anomaly_segments(frame_indices, deviation_values, height=0.2, distance=90):
    """
    Detect anomaly segments based on peaks in the deviation curve
    
    Args:
        frame_indices: Sorted list of frame indices
        deviation_values: Corresponding deviation values
        height: Minimum height for peak detection
        distance: Minimum distance between peaks
        
    Returns:
        segments: List of anomaly segments with start, end, center, and score
    """
    # Find peaks in deviation values
    peak_indices, peak_properties = find_peaks(deviation_values, height=height, distance=distance)
    
    segments = []
    for i, peak_idx in enumerate(peak_indices):
        center_frame_idx = frame_indices[peak_idx]
        
        # Define segment window (241 frames centered on peak)
        start_idx = max(0, center_frame_idx - 120)
        end_idx = min(frame_indices[-1], center_frame_idx + 120)
        
        # Add segment with metadata
        segments.append({
            "start_frame": start_idx,
            "end_frame": end_idx,
            "center": center_frame_idx,
            "score": deviation_values[peak_idx]
        })
    
    return segments

In [ ]:
def generate_outputs(video_path, segments, frame_scores, deviation_curve, output_dir):
    """
    Generate output files including JSON and UCF-compatible text
    
    Args:
        video_path: Path to the input video
        segments: List of anomaly segments
        frame_scores: Dictionary mapping frame indices to prompt scores
        deviation_curve: Tuple of (frame_indices, deviation_values)
        output_dir: Directory to save output files
        
    Returns:
        json_path: Path to the generated JSON file
        txt_path: Path to the UCF-compatible text file
    """
    os.makedirs(output_dir, exist_ok=True)
    video_name = os.path.basename(video_path)
    video_class = video_name.split("_")[0].split("0")[0]  # Extract class from filename
    
    # 1. Generate JSON output
    json_data = {
        "video": video_name,
        "anomaly_segments": segments,
        "frame_scores": {str(k): v for k, v in frame_scores.items()},
        "deviation_curve": {
            "frame_indices": deviation_curve[0],
            "deviation_values": deviation_curve[1]
        }
    }
    
    json_path = os.path.join(output_dir, f"{os.path.splitext(video_name)[0]}.json")
    with open(json_path, 'w') as f:
        json.dump(json_data, f, indent=2)
    
    # 2. Generate UCF-compatible text line
    ucf_line = f"{video_name}  {video_class}  "
    
    if len(segments) >= 1:
        ucf_line += f"{segments[0]['start_frame']}  {segments[0]['end_frame']}  "
        
        if len(segments) >= 2:
            ucf_line += f"{segments[1]['start_frame']}  {segments[1]['end_frame']}"
        else:
            ucf_line += "-1  -1"
    else:
        ucf_line += "-1  -1  -1  -1"
    
    txt_path = os.path.join(output_dir, "Temporal_Anomaly_Annotation_GEN.txt")
    with open(txt_path, 'a') as f:
        f.write(ucf_line + "\n")
    
    # 3. Optional: Generate CSV for debugging visualization
    csv_path = os.path.join(output_dir, "soft_labels.csv")
    with open(csv_path, 'a') as f:
        for frame_idx, scores in frame_scores.items():
            f.write(f"{video_name},{frame_idx},{scores['normal']},{scores['anomaly']},{scores['attack']},{scores['suspicious']}\n")
    
    print(f"Generated outputs in {output_dir}")
    return json_path, txt_path

In [ ]:
def visualize_results(frame_indices, deviation_values, segments):
    """
    Visualize deviation curve and detected anomaly segments
    
    Args:
        frame_indices: Sorted list of frame indices
        deviation_values: Corresponding deviation values
        segments: List of anomaly segments
    """
    plt.figure(figsize=(15, 5))
    plt.plot(frame_indices, deviation_values)
    
    # Highlight anomaly segments
    for segment in segments:
        plt.axvspan(segment['start_frame'], segment['end_frame'], alpha=0.2, color='red')
        plt.axvline(x=segment['center'], color='green', linestyle='--')
    
    plt.title('Anomaly Deviation Curve')
    plt.xlabel('Frame')
    plt.ylabel('Deviation')
    plt.grid(True)
    plt.show()

## __Main Pipeline Function__

In [ ]:
def process_video(video_path, output_dir, model_name="ViT-B/32", visualize=True):
    """
    Main function to process a video and detect anomalies
    
    Args:
        video_path: Path to the input video
        output_dir: Directory to save output files
        model_name: Name of the CLIP model to use
        visualize: Whether to visualize results
        
    Returns:
        json_path: Path to the generated JSON file
        txt_path: Path to the UCF-compatible text file
    """
    print(f"Processing video: {video_path}")
    
    # Step 1: Extract frames
    frames_dir = extract_frames(video_path, output_dir)
    
    # Step 2: Load CLIP model
    model, preprocess = load_clip_model(model_name)
    
    # Step 3: Compute frame embeddings
    frame_embeddings, frame_paths = compute_frame_embeddings(frames_dir, model, preprocess)
    
    # Step 4: Define prompts and compute prompt scores
    prompts = [
        "normal scene",
        "anomaly happening",
        "a person attacking",
        "suspicious object"
    ]
    
    frame_scores = compute_prompt_scores(frame_embeddings, model, prompts)
    
    # Step 5: Calculate deviation
    frame_indices, deviation_values = calculate_deviation(frame_scores)
    
    # Step 6: Detect anomaly segments
    segments = detect_anomaly_segments(frame_indices, deviation_values)
    
    # Step 7: Generate outputs
    json_path, txt_path = generate_outputs(
        video_path, segments, frame_scores, (frame_indices, deviation_values), output_dir
    )
    
    # Step 8: Visualize results
    if visualize:
        visualize_results(frame_indices, deviation_values, segments)
    
    return json_path, txt_path

## __CLI Interface__

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="CLIP-Based Temporal Anomaly Inference")
    parser.add_argument("--video", required=True, help="Path to input video")
    parser.add_argument("--output_dir", required=True, help="Directory to save outputs")
    parser.add_argument("--model", default="ViT-B/32", choices=["ViT-B/32", "ViT-L/14"], help="CLIP model to use")
    parser.add_argument("--visualize", action="store_true", help="Visualize results")
    return parser.parse_args()

def cli_main():
    args = parse_args()
    process_video(args.video, args.output_dir, args.model, args.visualize)

if __name__ == "__main__":
    cli_main()

## __Example Usage__

In [ ]:
# Download a sample video for testing
#!wget -P /content/ https://archive.org/download/UCF_Crime_Dataset_Samples/UCF_Crime_Dataset_Samples.zip
#!unzip -q /content/UCF_Crime_Dataset_Samples.zip -d /content/UCF_Crime_Sample

In [ ]:
# Process a sample video from UCF-Crime dataset
# Note: Replace this with actual UCF-Crime video path
sample_video = "/content/UCF_Crime_Sample/Abuse028_x264.mp4"
output_dir = "/content/drive/MyDrive/outputs"

# Check if file exists
if os.path.exists(sample_video):
    json_path, txt_path = process_video(sample_video, output_dir, model_name="ViT-B/32", visualize=True)
else:
    print(f"Sample video {sample_video} not found. Please upload a UCF-Crime dataset video.")

## __Batch Processing__

In [ ]:
def process_batch(video_dir, output_dir, model_name="ViT-B/32"):
    """
    Process multiple videos in a directory
    
    Args:
        video_dir: Directory containing input videos
        output_dir: Directory to save outputs
        model_name: Name of the CLIP model to use
    """
    # Get all video files
    video_extensions = [".mp4", ".avi", ".mov"]
    video_paths = []
    
    for ext in video_extensions:
        video_paths.extend(list(Path(video_dir).glob(f"**/*{ext}")))
    
    if not video_paths:
        print(f"No videos found in {video_dir}")
        return
    
    print(f"Found {len(video_paths)} videos")
    
    # Process each video
    for video_path in video_paths:
        process_video(str(video_path), output_dir, model_name, visualize=False)
    
    print("Batch processing complete")

In [ ]:
# Example batch processing
# process_batch("/content/UCF_Crime_Sample", "/content/drive/MyDrive/outputs", model_name="ViT-B/32")

## __Running via Command Line__

To run this notebook as a script, save it as a Python file and use the following command:

```bash
python phase0_infer.py --video data/Abuse028_x264.mp4 --output_dir outputs/ --model ViT-B/32
```
